# SEC Document Service — Raw Disclosure Documents (market.sec.or.th)

List and **download the original documents** companies file with the Thai SEC's information-disclosure
system (IDISC): the financial-statement **Excel** package, **Form 56-1**, **Form 56-2**, Key Financial
Ratio, and MD&A — for any SET/mai-listed issuer.

**You will learn how to:**
- Resolve a symbol/name to its SEC company record
- **List the available years** per category, and filter to a subset
- **Download them all** (or a filtered subset) to a folder
- Inspect a document, download one and peek inside the zip, handle dead links
- Use the `SecCompany` facade

> ⚠️ **Notes:** this is a **different host** from the SET services (`market.sec.or.th`, an ASP.NET site).
> Dates are **dd/mm/yyyy**. To see the **full year history**, pass a wide `from_date`/`to_date` — the
> default fetch returns only a recent window. Some links are dead on the SEC side and return an HTML
> "file not found" page under HTTP 200 — the download service detects this and raises `FetchError`.

## 1. Setup

In [ ]:
import io
import tempfile
import zipfile
from pathlib import Path

from settfex.services.sec import (
    DocumentCategory,
    SecCompany,
    download_sec_document,
    download_sec_documents,
    get_sec_documents,
    resolve_company,
)

# In Jupyter, await works directly (no asyncio.run needed)
SYMBOL = "CPALL"

## 2. Resolve a company

Map a symbol or name to the SEC `uniqueIDReference` (the join key).

In [ ]:
company = await resolve_company(SYMBOL)
print(company.company_name, "->", company.unique_id, "| primary:", company.is_primary)

## 3. List documents — wide window to see full history

The listing returns a **`SecDocumentList`** (a plain `list[SecDocument]` with extra helpers). Pass a
**wide** date window so every year is included.

In [ ]:
docs = await get_sec_documents(SYMBOL, from_date="01/01/2010", to_date="31/12/2026")
print("type:", type(docs).__name__, "| is a list:", isinstance(docs, list), "| count:", len(docs))

## 4. Available years per category

`summary()` prints a ready-made block; `years_by_category()` / `available_years()` return the data.
(MD&A rows carry a *date*, not a reporting *year*, so they show no years.)

In [ ]:
print(docs.summary())
print()
print("years_by_category():", docs.years_by_category())
print("available_years('form_56_1'):", docs.available_years("form_56_1"))
print("categories present:", [c.value for c in docs.categories()])

## 5. Filter to a subset (by category and/or year)

`filter(...)` returns another `SecDocumentList`.

In [ ]:
only_561 = docs.filter(category="form_56_1")
print(f"form_56_1: {len(only_561)} docs")

fs_2025 = docs.filter(category=DocumentCategory.FINANCIAL_STATEMENT, year=2025)
print(f"financial statements for 2025: {len(fs_2025)} docs")

## 6. Inspect a `SecDocument`

In [ ]:
doc = next(
    d for d in docs if d.category == DocumentCategory.FINANCIAL_STATEMENT and d.file_kind == "zip"
)
for field, value in doc.model_dump().items():
    print(f"{field:16}: {value}")

## 7. Download one document — and peek inside the zip

`download_sec_document` returns a `DownloadedFile` (filename + raw `bytes`). A financial-statement
package is a zip containing the original `FINANCIAL_STATEMENTS.XLSX` plus the auditor report and notes.

In [ ]:
file = await download_sec_document(doc)
print(f"{file.filename}  ({file.size:,} bytes, {file.content_type})")
print("contents:", zipfile.ZipFile(io.BytesIO(file.content)).namelist())
# Save to disk with:  file.save("./sec_downloads")

## 8. Download them all — or a filtered subset

Pass the whole `SecDocumentList` to download everything, or a `filter(...)` subset. Downloads run
concurrently (bounded), and dead links are skipped by default (`continue_on_error=True`).

In [ ]:
subset = docs.filter(category="form_56_1")  # or: docs (everything), docs.filter(year=2025), ...
with tempfile.TemporaryDirectory() as tmp:
    got = await download_sec_documents(subset, dest_dir=tmp, max_concurrency=5)
    print(f"downloaded {len(got)} file(s); on disk: {[p.name for p in Path(tmp).glob('*')]}")

## 9. Dead links (soft 404)

Some links are dead on the SEC side and return an HTML "file not found" page under HTTP 200. The
download service detects this and raises `FetchError` (in a batch, such items are skipped by default).

In [ ]:
from settfex.exceptions import FetchError

f56_2 = await get_sec_documents(
    SYMBOL, types="form_56_2", from_date="01/01/2024", to_date="31/12/2026"
)
dead = next((d for d in f56_2 if "dat/annual" in (d.file_id or "")), None)
if dead:
    try:
        await download_sec_document(dead)
    except FetchError as exc:
        print("correctly raised:", str(exc)[:80])
else:
    print("no dead dat/annual link for this symbol right now")

## 10. The `SecCompany` facade

In [ ]:
sec = SecCompany("CPALL")
await sec.resolve()
recent = await sec.list_documents(types="form_56_1", from_date="01/01/2020", to_date="31/12/2026")
print("available 56-1 years:", recent.available_years())
# Download all of them:  await sec.download_all(recent, dest_dir="./out")

## Use Cases

- **Fundamental research / AI pipelines**: enumerate available years, then pull the original
  `FINANCIAL_STATEMENTS.XLSX` for a universe of symbols and parse it yourself.
- **Annual-report archives**: `docs.filter(category="form_56_1")` (or `form_56_2`) → download all.
- **Disclosure monitoring**: list recent documents on a schedule and download new filings.

**See also**: `docs/settfex/services/sec/financial_report.md` · SET Financial Statements (notebook 11)
for the *modeled* balance sheet/income/cash-flow · News (notebook 16) for SET company news.